In [2]:
# -*- coding: utf-8 -*-
import os
import re
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple

# === INPUTS ===
MAIN_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv"
YML_DIR  = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.1_Emulator_Params_By_Repo.csv"

# === DETECTION REGEXES ===

# Emulator presence signals (we only extract parameters if we see an emulator setup)
EMULATOR_SIGNALS = [
    r'uses:\s*reactivecircus/android-emulator-runner',          # GH action
    r'\bavdmanager\b',                                          # avdmanager create avd
    r'\bsdkmanager\b[^\n"]*(system-images;android-\d+[^"\n]*)', # sdkmanager system-images
    r'\bemulator\b[^\n]*\s(-avd|@)\S+',                         # emulator -avd / @AVD
    r'\bandroid-wait-for-emulator\b',
    r'\bcircle-android\s+wait-for-boot\b',
]

# Parameters (multiple patterns to catch common styles)
# API level
API_PATTERNS = [
    r'\bapi[-_ ]?level\s*:\s*(\d{2})',                                             # api-level: 30
    r'\bapiLevel\s*:\s*(\d{2})',                                                   # apiLevel: 30
    r'system-images;android-(\d{2})\b',                                            # system-images;android-30;...
]

# System image source / target (google_apis, google_apis_playstore, aosp variants)
SYSIMG_PATTERNS = [
    r'\btarget\s*:\s*(google_apis(?:_playstore)?)',
    r'system-images;android-\d{2};([a-z0-9_]+)',                                   # ...;google_apis;...
    r'\b(system[-_ ]?image(?:source)?)\b\s*:\s*(google_apis(?:_playstore)?|aosp[_-]?\w*)',
]

# ABI / architecture
ABI_PATTERNS = [
    r'\b(abi|arch)\s*:\s*(x86_64|x86|arm64[-_]?v8a|armeabi[-_]?v7a)\b',
    r'system-images;android-\d{2};[a-z0-9_]+;(x86_64|x86|arm64[-_]?v8a|armeabi[-_]?v7a)\b',
]

# Device name / hardware profile
DEVICE_NAME_PATTERNS = [
    r'\bdevice\s*:\s*([A-Za-z0-9_ \-]+)\b',                                         # device: pixel_5
    r'\bprofile\s*:\s*([A-Za-z0-9_ \-]+)\b',                                        # profile: pixel_5
    r'\b--device\s+"?([A-Za-z0-9_ \-]+)"?',                                         # avdmanager --device "pixel_5"
    r'\b(avd[-_ ]?name)\s*:\s*([A-Za-z0-9_ \-]+)\b',                                 # avd name: Pixel_5
]

# Utility: search list of regexes and return a list of found values (dedup, keep order)
def find_values(patterns: List[str], text: str, group_idx: int = None) -> List[str]:
    found: List[str] = []
    for pat in patterns:
        for m in re.finditer(pat, text, flags=re.I | re.M):
            if group_idx is not None:
                val = m.group(group_idx)
            else:
                # heuristically choose the last group if groups exist, else whole match
                val = m.group(m.lastindex or 0) if m.lastindex else m.group(0)
            val = (val or "").strip()
            if val and val not in found:
                found.append(val)
    return found

def find_param_values(text: str) -> Dict[str, List[str]]:
    # API level
    api_vals = []
    for pat in API_PATTERNS:
        for m in re.finditer(pat, text, flags=re.I | re.M):
            # prefer last numeric group if present
            g = None
            if m.lastindex:
                for gi in range(1, m.lastindex + 1):
                    if m.group(gi) and m.group(gi).isdigit():
                        g = m.group(gi)
                        break
            if g and g not in api_vals:
                api_vals.append(g)

    # System image / target
    sysimg_vals = find_values(SYSIMG_PATTERNS, text)

    # ABI
    abi_vals = find_values(ABI_PATTERNS, text)

    # Device name: patterns may have 1 or 2 capturing groups
    device_vals: List[str] = []
    for pat in DEVICE_NAME_PATTERNS:
        for m in re.finditer(pat, text, flags=re.I | re.M):
            if m.lastindex:
                # take the last non-empty group as the value
                v = None
                for gi in range(m.lastindex, 0, -1):
                    if m.group(gi) and m.group(gi).strip():
                        v = m.group(gi).strip()
                        break
                if v and v not in device_vals:
                    device_vals.append(v)

    return {
        "api_level": api_vals,
        "system_image": sysimg_vals,
        "abi": abi_vals,
        "device_name": device_vals,
    }

def has_emulator_setup(text: str) -> bool:
    return any(re.search(p, text, flags=re.I | re.M) for p in EMULATOR_SIGNALS)

def join_or_default(values: List[str]) -> str:
    if values:
        return "; ".join(values)
    return "default"  # not explicitly set in YAML (likely action/tool default)

def main():
    df = pd.read_csv(MAIN_CSV)
    df.columns = [c.strip().lower() for c in df.columns]

    # Ensure required columns
    for col in ["full_name", "instru_t_ci_signal"]:
        if col not in df.columns:
            raise KeyError(f"Required column '{col}' not found in {MAIN_CSV}")

    # Filter to repos with instru_t_ci_signal == True
    df_sig = df[df["instru_t_ci_signal"] == True].copy()

    # Build index of YAML files by repo key (prefix before "__")
    yml_dir = Path(YML_DIR)
    if not yml_dir.exists():
        raise FileNotFoundError(f"YAML directory not found: {YML_DIR}")

    # Map: repo_key -> list of file paths
    files_by_repo: Dict[str, List[Path]] = {}
    for f in yml_dir.iterdir():
        if not f.is_file():
            continue
        if f.suffix.lower() not in (".yml", ".yaml"):
            continue
        name = f.name.lower()
        if "__" not in name:
            continue
        repo_key = name.split("__", 1)[0]  # e.g., owner.repo from owner.repo__github_actions++file.yml
        files_by_repo.setdefault(repo_key, []).append(f)

    rows: List[Dict[str, str]] = []
    for _, row in df_sig.iterrows():
        full_name = row["full_name"]
        repo_key = str(full_name).lower().strip()

        api_vals_all: List[str] = []
        sysimg_vals_all: List[str] = []
        abi_vals_all: List[str] = []
        device_vals_all: List[str] = []
        emulator_seen = False

        for f in files_by_repo.get(repo_key, []):
            try:
                txt = f.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                continue

            # Only extract if emulator device setup is detected in this YAML
            if has_emulator_setup(txt):
                emulator_seen = True
                params = find_param_values(txt)
                api_vals_all.extend([v for v in params["api_level"] if v not in api_vals_all])
                sysimg_vals_all.extend([v for v in params["system_image"] if v not in sysimg_vals_all])
                abi_vals_all.extend([v for v in params["abi"] if v not in abi_vals_all])
                device_vals_all.extend([v for v in params["device_name"] if v not in device_vals_all])

        # If no emulator setup at all, skip (we only list repos where device setup is emulator)
        if not emulator_seen:
            continue

        rows.append({
            "full_name": full_name,
            "api_level":    join_or_default(api_vals_all),
            "system_image": join_or_default(sysimg_vals_all),
            "abi":          join_or_default(abi_vals_all),
            "device_name":  join_or_default(device_vals_all),
        })

    out_df = pd.DataFrame(rows).sort_values(by="full_name").reset_index(drop=True)
    Path(os.path.dirname(OUTPUT_CSV)).mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved: {OUTPUT_CSV}  (rows={len(out_df)})")
    print(out_df.head(20).to_string(index=False))


if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.1_Emulator_Params_By_Repo.csv  (rows=365)
                     full_name api_level system_image         abi       device_name
              4ertuk.audioview   default      default     default           default
               a-mabe.openhiit    35; 34      default      x86_64       pixel_6_pro
a914-gowtham.compose-ratingbar   default      default     default           default
       aakira.expandablelayout   default      default     default           default
  abdelaziz-mahdy.pytorch_lite   default      default      x86_64           Nexus 6
             ably.ably-flutter   default      default     default           default
               achep.acdisplay   default      default     default           default
                acterglobal.a3    23; 28      default x86_64; x86 Nexus 6; Nexus 5X
      activitywatch.aw-android   default  google_apis      x86_64           Nexus 6
   adammc331.androidstudyguid

In [ ]:
# minimum Parameters for Third Party testing Lab
#it search the yaml files but if a supporting file .sh or .json is referenced should check that file too
# -*- coding: utf-8 -*-
# -*- coding: utf-8 -*-
import os
import re
import json
import yaml
import pandas as pd
from pathlib import Path
from typing import Dict, List, Set

# === Inputs/Outputs ===
MAIN_CSV   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv"
YAML_DIR   = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.1_ThirdPartyLab_Params_By_Repo.csv"

# 🔎 Where to look for referenced scripts/configs inside cloned repos.
# Add root directories that contain your repo working copies (e.g., owner\repo or owner.repo folders).
REPO_SEARCH_ROOTS = [
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Cloned_All",
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Cloned_Sample",
]

# === Third-party lab detection keywords ===
THIRD_PARTY_HITS = [
    r'\bgcloud\s+firebase\s+test\s+android\s+run\b',  # Firebase Test Lab CLI
    r'\bflank\s+android\s+run\b',                    # Flank (FTL)
    r'\bbrowserstack\b',                             # BrowserStack
    r'\bsaucectl\b',                                 # Sauce Labs
    r'\bdevicefarm\b',                               # AWS Device Farm
    r'\bkobiton\b',                                  # Kobiton
    r'\bbitbar\b',                                   # BitBar
]

# === Minimum parameters to extract (regex bundles) ===
# capture group 1 should be the value
PARAM_PATTERNS = {
    # Target device identifier: Firebase 'model=Pixel2', BrowserStack "device": "Google Pixel 6", Sauce 'deviceName'
    "target_device": [
        r'\bmodel\s*[:=]\s*([A-Za-z0-9._-]+)',                           # model=Pixel2 / model: Pixel2
        r'["\']device["\']\s*:\s*["\']([^"\']+)["\']',                   # "device": "Google Pixel 6"
        r'["\']deviceName["\']\s*:\s*["\']([^"\']+)["\']',               # "deviceName": "Google Pixel 4"
    ],
    # OS / API Level: Firebase 'version=30', BrowserStack "os_version": "12.0"
    "os_version_api": [
        r'\bversion\s*[:=]\s*([0-9]{2,3})\b',                            # version=30
        r'["\']os[_-]?version["\']\s*:\s*["\']([0-9.]+)["\']',           # "os_version": "12.0"
        r'\bapi\s*level\s*[:=]\s*([0-9]{2,3})\b',                        # api level: 33
    ],
    # App binary
    "app_apk_path": [
        r'\bapp\s*[:=]\s*([^\s"\'\n]+?\.(?:apk|aab))',                   # app=..., app: ...
        r'["\']app["\']\s*:\s*["\']([^"\']+\.(?:apk|aab))["\']',         # "app": "path.apk"
    ],
    # Test binary
    "test_apk_path": [
        r'\btest\s*[:=]\s*([^\s"\'\n]+?\.(?:apk|zip))',                  # test=..., test: ...
        r'["\']test["\']\s*:\s*["\']([^"\']+\.(?:apk|zip))["\']',        # "test": "path.apk"
    ],
    # Auth / credentials
    "auth_credentials": [
        r'\bGOOGLE_APPLICATION_CREDENTIALS\s*[:=]\s*([^\s"\']+\.json)\b',
        r'\bBROWSERSTACK_USERNAME\b', r'\bBROWSERSTACK_ACCESS_KEY\b',
        r'\bSAUCE_USERNAME\b', r'\bSAUCE_ACCESS_KEY\b',
        r'\bAWS_ACCESS_KEY_ID\b', r'\bAWS_SECRET_ACCESS_KEY\b',
        r'\bKOBITON_(?:USERNAME|API_KEY)\b',
    ],
}

# We’ll follow these referenced file types from YAML
FOLLOWABLE_EXTS = (".sh", ".bash", ".bat", ".cmd", ".ps1", ".json", ".yml", ".yaml")

# --- helpers ---
def read_text(p: Path) -> str:
    try:
        return p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        try:
            return p.read_text(encoding="latin-1", errors="ignore")
        except Exception:
            return ""

def lower_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def detect_third_party(raw: str) -> bool:
    text = raw.lower()
    return any(re.search(pat, text) for pat in THIRD_PARTY_HITS)

# Detect variable/secret-style tokens (GitHub/GitLab/Azure styles)
VAR_TOKEN_RE = re.compile(
    r'(?:(?:\$|\$\{)\s*[A-Za-z_][A-Za-z0-9_]*\s*\}?|'           # $VAR or ${VAR}
    r'\${{\s*(?:secrets|env|vars|inputs)\.[^}]+}}|'              # ${{ secrets.KEY }}
    r'%\([A-Za-z_][A-Za-z0-9_]*\)s)'                             # %(VAR)s
)

def extract_params_from_text_with_tokens(raw: str, patterns_by_key: dict):
    """Return (values, tokens) dicts for each param key."""
    out_vals: Dict[str, List[str]] = {k: [] for k in patterns_by_key.keys()}
    out_tokens: Dict[str, List[str]] = {k: [] for k in patterns_by_key.keys()}

    # explicit literals
    for key, patterns in patterns_by_key.items():
        for pat in patterns:
            for m in re.finditer(pat, raw, flags=re.I | re.M):
                val = m.group(1) if (m.lastindex and m.group(1)) else m.group(0)
                if not val:
                    continue
                val = val.strip().strip('"').strip("'")
                if val and val not in out_vals[key]:
                    out_vals[key].append(val)

    # variable tokens near param-related text (small windows)
    for key in patterns_by_key.keys():
        window_re = re.compile(rf'(?i)(?:{key}|app|test|device|model|version|os[_-]?version)[^\n]{{0,160}}')
        for w in window_re.finditer(raw):
            segment = raw[w.start():w.end()]
            for tm in VAR_TOKEN_RE.finditer(segment):
                tok = tm.group(0)
                if tok not in out_tokens[key]:
                    out_tokens[key].append(tok)

    return out_vals, out_tokens

def merge_values(acc: Dict[str, List[str]], new: Dict[str, List[str]]) -> None:
    for k, arr in new.items():
        for v in arr:
            if v not in acc[k]:
                acc[k].append(v)

def finalize_value_and_status(values_list: List[str], tokens_list: List[str]):
    """Return (value_str, status) according to what we actually found."""
    if values_list:
        return "; ".join(values_list), "explicit"
    if tokens_list:
        return "; ".join(tokens_list), "env_or_input"
    return "", "unspecified"

# extract file references from CI YAML lines that look like run/script/command
RUN_LINE = re.compile(r'(?mi)^\s*(?:run|script|command)\s*:\s*(.+)$')
PATH_REF  = re.compile(r'(?P<path>(?:\.{0,2}/|[A-Za-z]:\\)?[A-Za-z0-9._\-/\\]+(?:' + '|'.join([re.escape(e) for e in FOLLOWABLE_EXTS]) + r'))')

def find_file_refs_from_yaml(content: str) -> List[str]:
    refs: List[str] = []
    # single-line run/script/command
    for m in RUN_LINE.finditer(content):
        line = m.group(1)
        for ref in PATH_REF.finditer(line):
            refs.append(ref.group("path"))
    # naive scan for quoted config refs (e.g., -c ./.sauce/config.yml)
    for ref in re.findall(r'["\']([^"\']+\.(?:json|ya?ml|sh|bat|cmd|ps1))["\']', content, flags=re.I):
        refs.append(ref)
    # de-dup preserve order
    out, seen = [], set()
    for r in refs:
        r_norm = r.strip().strip('"').strip("'")
        if r_norm not in seen:
            seen.add(r_norm); out.append(r_norm)
    return out

def repo_key_from_full_name(full_name: str) -> str:
    return full_name.lower().replace("/", ".")

def find_repo_files(repo_key: str, roots: List[str], rel_path: str) -> List[Path]:
    """Try to locate a referenced file inside possible repo roots."""
    cand: List[Path] = []
    rel_norm = rel_path.replace("\\", "/").lstrip("./")
    for root in roots:
        owner_repo = repo_key.split(".", 1)
        variants = []
        if len(owner_repo) == 2:
            variants.append(Path(root) / owner_repo[0] / owner_repo[1] / rel_norm)
        variants.append(Path(root) / repo_key / rel_norm)
        for p in variants:
            if p.exists() and p.is_file():
                cand.append(p)
    return cand

def collect_yaml_index(yaml_dir: Path) -> Dict[str, List[Path]]:
    """Map: repo_key -> [yaml_file_paths] by filename prefix before '__'"""
    index: Dict[str, List[Path]] = {}
    for p in yaml_dir.iterdir():
        if p.is_file() and p.suffix.lower() in (".yml", ".yaml"):
            name = p.name.lower()
            if "__" in name:
                repo_key = name.split("__", 1)[0]
                index.setdefault(repo_key, []).append(p)
    return index

def main():
    # Load main CSV
    df = pd.read_csv(MAIN_CSV)
    df = lower_cols(df)

    # Normalize instru_t_ci_signal to boolean
    sig_col = "instru_t_ci_signal"
    if sig_col not in df.columns:
        raise KeyError(f"Required column '{sig_col}' not found in {MAIN_CSV}")

    def to_bool(x):
        if isinstance(x, str):
            return x.strip().lower() in ("true", "yes", "1")
        return bool(x)

    df["__signal"] = df[sig_col].apply(to_bool)
    df_sig = df[df["__signal"] == True].copy()

    yaml_dir = Path(YAML_DIR)
    if not yaml_dir.exists():
        raise FileNotFoundError(f"YAML directory not found: {YAML_DIR}")

    yaml_index = collect_yaml_index(yaml_dir)

    rows = []
    for _, rec in df_sig.iterrows():
        full_name = str(rec.get("full_name", "")).strip()
        if not full_name:
            continue
        repo_key = repo_key_from_full_name(full_name)

        # Accumulators per repo
        found_any_third_party = False
        acc_vals: Dict[str, List[str]]   = {k: [] for k in PARAM_PATTERNS.keys()}
        acc_tokens: Dict[str, List[str]] = {k: [] for k in PARAM_PATTERNS.keys()}
        provenance: Set[str] = set()

        for yml in yaml_index.get(repo_key, []):
            ytxt = read_text(yml)
            if not ytxt:
                continue

            # Third-party signals in YAML?
            if detect_third_party(ytxt):
                found_any_third_party = True
                vals, toks = extract_params_from_text_with_tokens(ytxt, PARAM_PATTERNS)
                merge_values(acc_vals, vals)
                merge_values(acc_tokens, toks)
                provenance.add(str(yml))

            # Follow referenced files from YAML (scripts/configs)
            for ref in find_file_refs_from_yaml(ytxt):
                # Locate in the repo clones
                refs = []
                for base in REPO_SEARCH_ROOTS:
                    refs.extend(find_repo_files(repo_key, [base], ref))

                for rp in refs:
                    rtxt = read_text(rp)
                    if not rtxt:
                        continue

                    if rp.suffix.lower() == ".json":
                        try:
                            j = json.loads(rtxt)
                            jtxt = json.dumps(j)
                            if detect_third_party(jtxt):
                                found_any_third_party = True
                            vals, toks = extract_params_from_text_with_tokens(jtxt, PARAM_PATTERNS)
                            merge_values(acc_vals, vals)
                            merge_values(acc_tokens, toks)
                            provenance.add(str(rp))
                            continue
                        except Exception:
                            pass

                    if rp.suffix.lower() in (".yml", ".yaml"):
                        try:
                            y = yaml.safe_load(rtxt)
                            ytxt2 = json.dumps(y, default=str)
                            if detect_third_party(ytxt2):
                                found_any_third_party = True
                            vals, toks = extract_params_from_text_with_tokens(ytxt2, PARAM_PATTERNS)
                            merge_values(acc_vals, vals)
                            merge_values(acc_tokens, toks)
                            provenance.add(str(rp))
                            continue
                        except Exception:
                            pass

                    if rp.suffix.lower() in (".sh", ".bash", ".bat", ".cmd", ".ps1"):
                        if detect_third_party(rtxt):
                            found_any_third_party = True
                        vals, toks = extract_params_from_text_with_tokens(rtxt, PARAM_PATTERNS)
                        merge_values(acc_vals, vals)
                        merge_values(acc_tokens, toks)
                        provenance.add(str(rp))

        # Only include repos where third-party lab usage was detected anywhere we looked
        if found_any_third_party:
            # finalize each parameter value + status
            target_device, target_status     = finalize_value_and_status(acc_vals["target_device"],   acc_tokens["target_device"])
            os_version_api, os_status        = finalize_value_and_status(acc_vals["os_version_api"],  acc_tokens["os_version_api"])
            app_apk_path, app_status         = finalize_value_and_status(acc_vals["app_apk_path"],    acc_tokens["app_apk_path"])
            test_apk_path, test_status       = finalize_value_and_status(acc_vals["test_apk_path"],   acc_tokens["test_apk_path"])
            auth_credentials, auth_status    = finalize_value_and_status(acc_vals["auth_credentials"], acc_tokens["auth_credentials"])

            rows.append({
                "full_name": full_name,
                "target_device": target_device,               "target_device_status": target_status,
                "os_version_api": os_version_api,             "os_version_api_status": os_status,
                "app_apk_path": app_apk_path,                 "app_apk_path_status": app_status,
                "test_apk_path": test_apk_path,               "test_apk_path_status": test_status,
                "auth_credentials": auth_credentials,         "auth_credentials_status": auth_status,
                "sources": "; ".join(sorted(provenance)) if provenance else "YAML only",
            })

    out_df = pd.DataFrame(rows).sort_values("full_name").reset_index(drop=True)
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved: {OUTPUT_CSV} (rows={len(out_df)})")
    if not out_df.empty:
        print(out_df.head(15).to_string(index=False))

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ThirdPartyLab_Params_By_Repo.csv (rows=25)
                           full_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                target_device target_device_status               